# Multi-Agent Customer Support Automation

> **Built with:** CrewAI · OpenAI GPT · ScrapeWebsiteTool  
> **Author:** Pradeep Kumar

---

## 🎯 What This Project Does

This project implements a **two-agent customer support system** using CrewAI. Instead of a single LLM handling a support request end-to-end, two specialised agents collaborate:

1. **Support Agent** — Researches the customer's inquiry, scrapes official documentation, and drafts a comprehensive response.
2. **QA Agent** — Reviews the draft for accuracy, completeness, and tone before the response reaches the customer.

This pattern mirrors how real support teams operate: a first responder handles the inquiry, and a senior reviewer ensures quality before dispatch.

---

## 🏗️ System Architecture

```
Customer Inquiry
      │
      ▼
┌─────────────────────────┐
│   Support Agent          │  role: Senior Support Representative
│   ─────────────────────  │  tools: ScrapeWebsiteTool (docs)
│   • Reads inquiry        │  allow_delegation: False
│   • Scrapes docs         │  memory: shared via Crew
│   • Drafts response      │
└───────────┬─────────────┘
            │  passes draft
            ▼
┌─────────────────────────┐
│   QA Agent               │  role: Support Quality Assurance Specialist
│   ─────────────────────  │  tools: none (review only)
│   • Reviews draft        │  allow_delegation: True (can send back)
│   • Checks accuracy      │  memory: shared via Crew
│   • Finalises tone       │
└───────────┬─────────────┘
            │
            ▼
    Final Customer Response
```

---

## 🧠 Key Concepts Demonstrated

| Concept | Implementation |
|---|---|
| **Role Playing** | Each agent has a distinct `role`, `goal`, and `backstory` |
| **Focus** | Agents are prompted to stay in character and avoid assumptions |
| **Tool Use** | Support Agent uses `ScrapeWebsiteTool` to ground answers in official docs |
| **Cooperation** | QA Agent can delegate tasks back to the Support Agent if needed |
| **Guardrails** | Task `expected_output` constrains response scope and format |
| **Memory** | `memory=True` on the Crew enables agents to share context across tasks |

---

## ⚙️ Setup & Prerequisites

### Installation

```bash
pip install crewai crewai-tools python-dotenv
```

### API Keys

Create a `.env` file in the project root (see `.env.example`):

```
OPENAI_API_KEY=your_openai_api_key_here
```

> ⚠️ Never commit your `.env` file. It is already listed in `.gitignore`.

---
## 1. Environment Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Load API keys from .env file (never hard-code keys)
from dotenv import load_dotenv
load_dotenv()

# Verify the key is loaded (will print None if .env is missing)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Check your .env file."

# Set the model — swap to gpt-4o for better reasoning on complex inquiries
os.environ["OPENAI_MODEL_NAME"] = "gpt-3.5-turbo"

print("✅ Environment loaded successfully.")

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import ScrapeWebsiteTool

---
## 2. Define the Tools

The `ScrapeWebsiteTool` lets an agent fetch and read a specific URL at runtime.
Here we point it at the official CrewAI documentation so the Support Agent can look up accurate answers rather than hallucinating.

**Design note:** Grounding agents in real documentation is one of the most effective guardrails against hallucination in customer-facing systems.

In [ ]:
# Tool scoped to the official CrewAI documentation page
# In production you would point this at your own product docs or a RAG retrieval tool
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

---
## 3. Define the Agents

Each agent is configured with three identity properties:
- **`role`** — the job title, which focuses the LLM's persona
- **`goal`** — the outcome the agent is optimising for
- **`backstory`** — contextual background that shapes how the agent approaches its work

### Agent 1 — Support Agent

Handles the inquiry directly. `allow_delegation=False` means this agent cannot hand off its task — it must produce the draft itself.

In [ ]:
support_agent = Agent(
    role="Senior Support Representative",
    goal="Be the most friendly and helpful support representative in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and are now working on providing "
        "support to {customer}, a super important customer for your company. "
        "You need to make sure that you provide the best support possible. "
        "Make sure to provide full, complete answers and make no assumptions."
    ),
    allow_delegation=False,  # Must handle the inquiry itself — no passing the buck
    verbose=True
)

### Agent 2 — Quality Assurance Agent

Reviews the Support Agent's draft. `allow_delegation` defaults to `True`, meaning this agent *can* send the draft back to the Support Agent with revision requests if it finds gaps.

This creates a feedback loop between the two agents — a simple form of multi-agent reflection.

In [ ]:
support_quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the best support quality assurance in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and are now working with your team "
        "on a request from {customer}, ensuring that the support representative is "
        "providing the best support possible.\n"
        "You need to make sure that the support representative is providing full, "
        "complete answers and making no assumptions."
    ),
    # allow_delegation defaults to True — this agent CAN send work back for revision
    verbose=True
)

---
## 4. Define the Tasks

Tasks tell agents *what to do* and *what good output looks like*. The `expected_output` field acts as a guardrail — it defines the format and completeness criteria the LLM must meet.

**Tool assignment:** Tools can be assigned at agent level (available for all tasks) or task level (only for that specific task). Here we assign `docs_scrape_tool` at task level — the Support Agent only uses it during the inquiry resolution task, not during any other work.

### Task 1 — Inquiry Resolution

In [ ]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
        "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
        "Make sure to use everything you know to provide the best support possible. "
        "You must strive to provide a complete and accurate response to the customer's inquiry."
    ),
    expected_output=(
        "A detailed, informative response to the customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references to everything used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, leaving no questions unanswered, "
        "and maintain a helpful and friendly tone throughout."
    ),
    tools=[docs_scrape_tool],   # Task-level tool: only active during this task
    agent=support_agent,
)

### Task 2 — Quality Assurance Review

The QA Agent does not use any tools — it reviews the draft produced by Task 1 using reasoning alone. It checks for accuracy, completeness, tone, and unresolved questions.

In [ ]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
        "high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry have been addressed thoroughly, "
        "with a helpful and friendly tone.\n"
        "Check for references and sources used to find the information, "
        "ensuring the response is well-supported and leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response ready to be sent to the customer.\n"
        "This response should fully address the customer's inquiry, incorporating all "
        "relevant feedback and improvements.\n"
        "Don't be too formal — we are a chill and cool company — "
        "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
    # No tools assigned — QA is a reasoning-only review task
)

---
## 5. Assemble the Crew

The `Crew` object wires agents and tasks together into an executable pipeline.

`memory=True` enables **shared memory across all agents in the crew**. This means:
- Short-term memory: agents can reference earlier steps in the same run
- Long-term memory (with embedding storage): patterns persist across runs
- Entity memory: the crew remembers key entities (people, companies) mentioned

For a customer support system, memory is essential — it allows the QA Agent to fully understand the context of the Support Agent's research.

In [ ]:
crew = Crew(
    agents=[support_agent, support_quality_assurance_agent],
    tasks=[inquiry_resolution, quality_assurance_review],
    verbose=True,
    memory=True  # Enables shared context across agents and tasks
)

---
## 6. Run the Crew

We pass in a sample inquiry using CrewAI's **dynamic input system** (`{customer}`, `{person}`, `{inquiry}` are template variables defined in the task descriptions above).

This makes the crew reusable — you can swap in any customer and inquiry without touching the agent or task definitions.

In [ ]:
# Sample input — swap these values to test with different customers/inquiries
inputs = {
    "customer": "AI Corp.",
    "person": "Pradeep K",
    "inquiry": (
        "I need help with setting up a Crew and kicking it off. "
        "Specifically, how can I add memory to my crew? "
        "Can you provide guidance?"
    )
}

result = crew.kickoff(inputs=inputs)

---
## 7. View the Final Response

In [ ]:
from IPython.display import Markdown
Markdown(result)

---
## 8. What I Learned & What I'd Build Next

### Key Takeaways

- **Role-based agent design** dramatically improves output quality over a single-agent approach. The QA Agent consistently caught gaps and improved tone that the Support Agent left unpolished.
- **Task-level tool scoping** is cleaner than agent-level in this pattern — the QA Agent has no need to scrape docs, so keeping the tool at task level avoids unnecessary LLM calls.
- **`allow_delegation=False`** on the Support Agent is an important guardrail. Without it, the agent could deflect the inquiry instead of researching it.
- **Memory** significantly improves QA accuracy on longer, multi-step inquiries by ensuring the reviewer has full context.

### Extensions I'd Implement

1. **RAG layer:** Replace `ScrapeWebsiteTool` with a RAG pipeline (LangChain + Pinecone/ChromaDB) so the Support Agent can query an entire documentation corpus, not just one page.
2. **Ticket classification agent:** Add a triage agent upstream that categorises inquiries (billing, technical, feature request) and routes to specialist sub-crews.
3. **Observability:** Instrument with LangSmith or CrewAI's built-in telemetry to trace agent reasoning steps and identify where hallucinations occur.
4. **Production API:** Wrap `crew.kickoff()` in a FastAPI endpoint so the system can serve real-time support requests.
5. **Human-in-the-loop:** Add a `HumanInputTool` step between draft and QA review for high-stakes inquiries, where a human can approve or redirect before the QA Agent finalises.

---


